# Multi-Geometry Stitching

Combine angle-separated detector images with one calibrated integrator
per image. Normalization is a per-image monitor factor, not a visual
adjustment; inspect overlap and correction assumptions before analysis.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from xrd_tools.integrate import create_multigeometry_integrators, stitch_1d, stitch_2d
from xrd_tools.viz import plot_1d


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
image_paths = []  # Real mode: ordered detector images.
poni_paths = []   # Real mode: one PONI per image.
monitor = None    # Optional one value per image.
q_range = widgets.FloatRangeSlider(value=(1.0, 4.0), min=0.5, max=6.0, step=0.05, description="q range", continuous_update=False)
compute = widgets.Button(description="Compute stitch", button_style="primary")
display(widgets.VBox([q_range, compute]))


In [ ]:
q = np.linspace(1.0, 4.0, 220)
left = 15 + 70 * np.exp(-0.5 * ((q - 2.15) / 0.10) ** 2)
right = 15 + 70 * np.exp(-0.5 * ((q - 2.15) / 0.10) ** 2) * 1.02
if not SMOKE_MODE:
    assert image_paths and len(image_paths) == len(poni_paths)
    integrators = create_multigeometry_integrators(poni_paths)
    stitched = stitch_1d(image_paths, integrators, radial_range=tuple(q_range.value), normalization=monitor)
    q, merged = stitched.radial, stitched.intensity
else:
    merged = (left + right) / 2
fig, ax = plt.subplots(figsize=(7, 3))
plot_1d(ax, q, merged, fmt="-", attrs={"xlabel": "q (A^-1)", "ylabel": "Intensity", "title": "Stitched pattern"})
plt.show()
